In [ ]:
from lerobot.datasets.lerobot_dataset import MultiLeRobotDataset
import time
st = time.time()
ds = MultiLeRobotDataset(
    repo_ids=["libero_merged_2k",],
    # root="/home/ubuntu/sereact_lerobot_data/sereact_picking8",
    root="/home/ubuntu/mount-point/il_datasets",
    # repo_ids=ids,
    # root=root,
    delta_timestamps = {
        "observation.state": [i/10 for i in range(-1,1)] + [i/10 for i in range(1, 50)],
        "action": [i/10 for i in range(50)],
        # "observation.subtask_instr": [i/10 for i in range(50)],
    },
    # cls=LeRobotPi05Dataset,
    # state_as_actions=False,
    # max_cameras=3,
    # predict_subtask=False,
    # predict_bbox=False,
    # predict_keypoints=False,
    # force_seg_input=True,
    # force_subtask_input=False,
)
print(f"Time taken: {time.time() - st} seconds")
data = ds[0]

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Loading datasets:   0%|          | 0/1 [00:00<?, ?dataset/s]

Loading dataset libero_merged_2k from /home/ubuntu/mount-point/il_datasets/libero_merged_2k


Resolving data files:   0%|          | 0/1618 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/142 [00:00<?, ?it/s]

In [ ]:
print(*data.keys(), sep="\n")

timestamp
observation.state
action
observation.subtask_instr
observation.task_instr
observation.images.static1
observation.masks.static1
observation.depths.static1
observation.intrinsics.static1
observation.images.wrist1
frame_index
episode_index
index
task_index
observation.state_is_pad
action_is_pad
task
dataset_info
dataset_index


In [ ]:
data["action"].shape

torch.Size([50, 7])

In [9]:
data["observation.task_instr"]

'Place the two mugs onto the saucers.'

In [12]:
data["observation.images.static1"].shape

torch.Size([3, 256, 256])

In [16]:
ds[0]["action"][1]

tensor([ 0.0134,  0.0000, -0.0000,  0.0000,  0.0000, -0.0000, -1.0000])

In [17]:
ds[1]["action"][0]

tensor([ 0.0134,  0.0000, -0.0000,  0.0000,  0.0000, -0.0000, -1.0000])

In [16]:
# Build a metadata DataFrame without storing large tensors (images, depths, etc.)
import pandas as pd
import torch
from tqdm.notebook import tqdm

LIGHT_TOP_LEVEL_KEYS = [
    "timestamp","dataset_index","frame_index","episode_index",
    "index","task_index","task"
]

def scalar_or_list(v):
    if isinstance(v, torch.Tensor):
        if v.numel() == 1:
            return v.item()
        return v.tolist()
    return v

def extract_metadata(sample):
    out = {}
    # Simple scalars
    for k in LIGHT_TOP_LEVEL_KEYS:
        if k in sample:
            out[k] = scalar_or_list(sample[k])
    # Fallback for task string
    if "task" not in out and "observation.task_instr" in sample:
        out["task"] = sample["observation.task_instr"]
    # Padding stats
    for pad_key in ["observation.state_is_pad","action_is_pad"]:
        if pad_key in sample and isinstance(sample[pad_key], torch.Tensor):
            t = sample[pad_key] == True
            out[f"{pad_key}.true_count"] = int(t.sum().item())
            out[f"{pad_key}.length"] = int(sample[pad_key].numel())
    # dataset_info flatten (only primitive types)
    di = sample.get("dataset_info", {})
    if isinstance(di, dict):
        for k,v in di.items():
            if isinstance(v, (str,int,float,bool)):
                out[f"dataset_info.{k}"] = v
    return out

records = []
i = 0
sample = ds[i]
while sample["episode_index"] == 0:
    sample = ds[i]
    records.append(extract_metadata(sample))
    if (i+1) % 200 == 0:
        print(f"Processed {i+1}/{len(ds)} samples")

    i += 1

metadata_df = pd.DataFrame(records)
print("DataFrame shape:", metadata_df.shape)
display(metadata_df.head(), metadata_df.tail())

# Optional: save
metadata_df.to_csv("lerobot_metadata.csv", index=False)
print("Saved lerobot_metadata.csv")

Processed 200/263890 samples
DataFrame shape: (215, 24)


,timestamp,dataset_index,frame_index,episode_index,index,task_index,task,observation.state_is_pad.true_count,observation.state_is_pad.length,action_is_pad.true_count,...,dataset_info.adjusted_gripper,dataset_info.stereo_replace_depth,dataset_info.fake_stereo,dataset_info.action_type,dataset_info.robot_embodiment,dataset_info.robot_type,dataset_info.force_seg_input,dataset_info.episode_index,dataset_info.is_last_frame_of_episode,dataset_info.no_state
0,0.0,0,0,0,0,0,Place the two mugs onto the saucers.,1,51,0,...,False,False,False,joint_state,single_arm,franka,False,0,False,False
1,0.1,0,1,0,1,0,Place the two mugs onto the saucers.,0,51,0,...,False,False,False,joint_state,single_arm,franka,False,0,False,False
2,0.2,0,2,0,2,0,Place the two mugs onto the saucers.,0,51,0,...,False,False,False,joint_state,single_arm,franka,False,0,False,False
3,0.3,0,3,0,3,0,Place the two mugs onto the saucers.,0,51,0,...,False,False,False,joint_state,single_arm,franka,False,0,False,False
4,0.4,0,4,0,4,0,Place the two mugs onto the saucers.,0,51,0,...,False,False,False,joint_state,single_arm,franka,False,0,False,False


,timestamp,dataset_index,frame_index,episode_index,index,task_index,task,observation.state_is_pad.true_count,observation.state_is_pad.length,action_is_pad.true_count,...,dataset_info.adjusted_gripper,dataset_info.stereo_replace_depth,dataset_info.fake_stereo,dataset_info.action_type,dataset_info.robot_embodiment,dataset_info.robot_type,dataset_info.force_seg_input,dataset_info.episode_index,dataset_info.is_last_frame_of_episode,dataset_info.no_state
210,21.000000,0,210,0,210,0,Place the two mugs onto the saucers.,46,51,46,...,False,False,False,joint_state,single_arm,franka,False,0,False,False
211,21.100000,0,211,0,211,0,Place the two mugs onto the saucers.,47,51,47,...,False,False,False,joint_state,single_arm,franka,False,0,False,False
212,21.200001,0,212,0,212,0,Place the two mugs onto the saucers.,48,51,48,...,False,False,False,joint_state,single_arm,franka,False,0,False,False
213,21.299999,0,213,0,213,0,Place the two mugs onto the saucers.,49,51,49,...,False,False,False,joint_state,single_arm,franka,False,0,True,False
214,0.000000,0,0,1,214,1,Place the grey mug on the saucer.,1,51,0,...,False,False,False,joint_state,single_arm,franka,False,1,False,False


Saved lerobot_metadata.csv


In [1]:
from libero.libero import benchmark
bd = benchmark.get_benchmark_dict()
suite = bd["libero_goal"]()
for i in range(suite.n_tasks):
    t = suite.get_task(i)
    # if "two mugs" in t.language.lower() or ("white mug" in t.language.lower() and "yellow" in t.language.lower() and "plate" in t.language.lower()):
    #     print("task_id:", i, "| language:", t.language)

    print("task_id:", i, "| language:", t.language)

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
task_id: 0 | language: open the middle drawer of the cabinet
task_id: 1 | language: put the bowl on the stove
task_id: 2 | language: put the wine bottle on top of the cabinet
task_id: 3 | language: open the top drawer and put the bowl inside
task_id: 4 | language: put the bowl on top of the cabinet
task_id: 5 | language: push the plate to the front of the stove
task_id: 6 | language: put the cream cheese in the bowl
task_id: 7 | language: turn on the stove
task_id: 8 | language: put the bowl on the plate
task_id: 9 | language: put the wine bottle on the rack


In [20]:
type(sample["action"])

torch.Tensor

In [22]:
sample.keys()

dict_keys(['timestamp', 'observation.state', 'action', 'observation.subtask_instr', 'observation.task_instr', 'observation.images.static1', 'observation.masks.static1', 'observation.depths.static1', 'observation.intrinsics.static1', 'observation.images.wrist1', 'frame_index', 'episode_index', 'index', 'task_index', 'observation.state_is_pad', 'action_is_pad', 'task', 'dataset_info', 'dataset_index'])

In [24]:
# Extract episode 0 actions and static1 frames; save actions + MP4 video
import torch, imageio, numpy as np

episode_actions = []
episode_frames = []
for idx in tqdm(range(300)):
    s = ds[idx]
    if s['episode_index'] != 0:
        break
    # action tensor
    episode_actions.append(s['action'].detach().cpu())
    # image frame (C,H,W) assumed RGB uint8 or float in [0,1]
    frame = s['observation.images.static1']
    if isinstance(frame, torch.Tensor):
        frame = frame.detach().cpu()
    episode_frames.append(frame)

print(f"Collected {len(episode_actions)} steps for episode 0")
actions_tensor = torch.stack(episode_actions)  # (T, action_dim[...])
torch.save(actions_tensor, 'episode0_actions.pt')
print('Saved actions -> episode0_actions.pt', actions_tensor.shape)

frames_tensor = torch.stack(episode_frames)  # (T,C,H,W)
vid = frames_tensor
if vid.dtype != torch.uint8:
    if vid.dtype.is_floating_point:
        vid = (vid.clamp(0,1)*255).to(torch.uint8)
    else:
        vid = vid.to(torch.uint8)
vid_np = vid.permute(0,2,3,1).numpy()  # (T,H,W,C)
imageio.mimsave('episode0_static1.mp4', list(vid_np), fps=30)
print('Saved video -> episode0_static1.mp4 (fps=30)')

# Example reload for replay logic
loaded_actions = torch.load('episode0_actions.pt')
print('Reloaded actions shape:', loaded_actions.shape)

  0%|          | 0/300 [00:00<?, ?it/s]

Collected 214 steps for episode 0
Saved actions -> episode0_actions.pt torch.Size([214, 50, 7])
Saved video -> episode0_static1.mp4 (fps=30)
Reloaded actions shape: torch.Size([214, 50, 7])
Saved video -> episode0_static1.mp4 (fps=30)
Reloaded actions shape: torch.Size([214, 50, 7])


In [1]:
import torch 
loaded_actions = torch.load('episode0_actions.pt')

In [3]:
loaded_actions[0][0]

tensor([ 0.0161,  0.0000, -0.0000,  0.0000,  0.0000, -0.0000, -1.0000])

In [5]:
10700 / 10 / 60

17.833333333333332